# Workshop 2

## Задача 1 – `memoize`

**`@memoize`**
* кешира резултатите на функцията по позиционните ѝ аргументи (приемаме, че са hashable)
* атрибути `hits` и `misses` на декорираната функция – колко пъти резултатът е взет от кеша и колко пъти е сметнат наново
* метод `cache_clear()`, който изчиства кеша и нулира броячите
* декорираната функция пази името си (`__name__`)


In [ ]:
def memoize(func):
    pass  # your code here


# Tests:

@memoize
def slow_fib(n):
    return n if n < 2 else slow_fib(n - 1) + slow_fib(n - 2)

assert slow_fib(30) == 832040
assert (slow_fib.misses, slow_fib.hits) == (31, 28)
assert slow_fib(30) == 832040 and slow_fib.hits == 29
slow_fib.cache_clear()
assert (slow_fib.misses, slow_fib.hits) == (0, 0)
slow_fib(2)
assert (slow_fib.misses, slow_fib.hits) == (3, 0), "the cache should really be empty after cache_clear"
assert slow_fib.__name__ == "slow_fib"


"✅ All OK!"

## Задача 2 – Мързеливи заявки

Напишете клас `Query(iterable)`, с който да строим вериги от операции върху поредици:

```python
Query(range(100)).where(lambda x: x % 3 == 0).select(lambda x: x * x).skip(1).take(3).to_list()
# [9, 36, 81]
```

**Операции** – всяка връща **нов** `Query`:
* `where(predicate)` – само елементите, за които `predicate` е истина
* `select(func)` – прилага `func` върху всеки елемент
* `take(n)` / `skip(n)` – първите `n` елемента / всички без първите `n`
* `distinct(key=None)` – премахва повторенията (сравнява по `key(x)`, ако е даден), пазейки първата поява
* `chunk(size)` – групира елементите в списъци с дължина `size` (последният може да е по-къс)
* `flatten()` – "разгъва" едно ниво на вложеност: `[[1, 2], [3]]` → `1, 2, 3`

**Крайни операции:**
* `for x in query`
* `to_list()`, `count()`
* `first(default=None)` – първият елемент или `default`, ако няма такъв

⚠️ Всичко трябва да е **мързеливо** – нищо не се изчислява, докато не започнем да итерираме, и се изчислява само толкова, колкото е нужно. Това означава, че трябва да работи и с безкрайни поредици (`itertools.count()`).

In [ ]:
class Query:
    pass
# Tests:

import itertools

assert Query(range(100)).where(lambda x: x % 3 == 0).select(lambda x: x * x).skip(1).take(3).to_list() == [9, 36, 81]
assert Query([3, 1, 3, 2, 1]).distinct().to_list() == [3, 1, 2]
assert Query(["apple", "Avocado", "banana", "Blueberry", "cherry"]).distinct(key=lambda s: s[0].lower()).to_list() == ["apple", "banana", "cherry"]
assert Query(range(7)).chunk(3).to_list() == [[0, 1, 2], [3, 4, 5], [6]]
assert Query([[1, 2], [], [3]]).flatten().to_list() == [1, 2, 3]
assert Query("hello").chunk(2).select("".join).to_list() == ["he", "ll", "o"]
assert Query(range(10)).where(lambda x: x > 100).first() is None
assert Query(range(10)).where(lambda x: x > 100).first(default=-1) == -1
assert Query(range(10)).skip(3).count() == 7
assert [x for x in Query("abc").select(str.upper)] == ["A", "B", "C"]

# each operation returns a new Query, the original stays the same
numbers = Query(range(5))
evens = numbers.where(lambda x: x % 2 == 0)
assert isinstance(evens, Query) and evens is not numbers
assert numbers.to_list() == [0, 1, 2, 3, 4]
assert evens.to_list() == [0, 2, 4]

# laziness
calls = []
def spy(x):
    calls.append(x)
    return x

q = Query(itertools.count()).select(spy).where(lambda x: x % 2).take(3)
assert calls == [], "nothing should be computed before iterating"
assert q.to_list() == [1, 3, 5]
assert calls == [0, 1, 2, 3, 4, 5], "only as much as needed should be computed"

assert Query(itertools.count(1)).select(lambda x: [x] * x).flatten().chunk(4).skip(2).first() == [4, 4, 5, 5]
assert Query(itertools.count()).select(lambda x: x % 5).distinct().take(5).to_list() == [0, 1, 2, 3, 4]

"✅ All OK!"

AttributeError: 'NoneType' object has no attribute 'to_list'

## Задача 3 – Символни изрази

Израз представяме като дърво:
* число (`int` или `float`)
* променлива – низ, напр. `"x"`
* `(op, a, b)`, където `op` е `"+"`, `"-"`, `"*"` или `"/"`, а `a` и `b` са изрази
* `("neg", a)` – отрицание на израза `a`

Например $ 2 \cdot x + (-y) $ е `("+", ("*", 2, "x"), ("neg", "y"))`.

Напишете следните функции, като използвате **`match`**:

**`evaluate(expr, env)`** – пресмята израза, като стойностите на променливите са в речника `env`. Връща `None`, ако изразът е невалиден, има непозната променлива или деление на 0.

**`to_string(expr)`** – връща израза като низ. Всяка бинарна операция е в скоби: `"((2 * x) + -y)"`. Отрицанието е `"-"` пред израза.

**`simplify(expr)`** – опростява израза рекурсивно (първо децата, после текущия възел) по следните правила:
* операция само върху числа се пресмята (с изключение на деление на 0 – тогава изразът остава какъвто е)
* `x + 0`, `0 + x`, `x - 0`, `x * 1`, `1 * x`, `x / 1` → `x`
* `x * 0`, `0 * x`, `0 / x` → `0`
* `x - x` → `0` и `x / x` → `1` (когато двете страни са еднакви изрази)
* `0 - x` → `("neg", x)`
* `("neg", ("neg", x))` → `x`

In [ ]:
def evaluate(expr, env):
    pass  # your code here


def to_string(expr):
    pass  # your code here


def simplify(expr):
    pass  # your code here


# Tests:

expr = ("+", ("*", 2, "x"), ("neg", "y"))
env = {"x": 5, "y": 3}

assert evaluate(42, {}) == 42
assert evaluate("x", env) == 5
assert evaluate(expr, env) == 7
assert evaluate(("/", "x", 2), env) == 2.5
assert evaluate(("-", ("neg", "x"), ("*", "y", "y")), env) == -14
assert evaluate("z", env) is None, "unknown variable"
assert evaluate(("/", 1, ("-", "x", 5)), env) is None, "division by zero"
assert evaluate(("%", 1, 2), env) is None, "unknown operation"
assert evaluate(("+", 1), env) is None, "wrong number of arguments"
assert evaluate(("+", 1, ("+", "x", "z")), env) is None, "errors should propagate"

assert to_string(expr) == "((2 * x) + -y)"
assert to_string(("neg", ("-", "a", 1.5))) == "-(a - 1.5)"
assert to_string("x") == "x" and to_string(7) == "7"

assert simplify(("+", 2, 3)) == 5
assert simplify(("*", ("+", 1, 1), ("-", 10, 7))) == 6
assert simplify(("+", "x", 0)) == "x"
assert simplify(("*", 1, ("+", "x", ("*", "y", 0)))) == "x"
assert simplify(("*", ("+", "x", "y"), ("-", 3, 3))) == 0
assert simplify(("-", ("+", "x", 1), ("+", "x", 1))) == 0
assert simplify(("/", ("*", "a", "b"), ("*", "a", "b"))) == 1
assert simplify(("-", 0, "x")) == ("neg", "x")
assert simplify(("neg", ("neg", ("+", "x", 0)))) == "x"
assert simplify(("neg", ("+", 2, 3))) == -5
assert simplify(("/", "x", ("-", 2, 2))) == ("/", "x", 0), "division by zero stays"
assert simplify(("+", ("*", 2, "x"), ("*", 3, 4))) == ("+", ("*", 2, "x"), 12)
assert simplify("x") == "x" and simplify(3.5) == 3.5

big = ("+", ("*", ("-", "x", 0), ("/", "y", "y")), ("-", 0, ("*", 0, "z")))
assert simplify(big) == "x"
assert evaluate(big, {"x": 4, "y": 2, "z": 9}) == evaluate(simplify(big), {"x": 4}) == 4

"✅ All OK!"

## Задача 4 – Отчет за продажби

Дадени са поръчки като `namedtuple`-и `Order(customer, product, quantity, price, month)`.

Напишете функциите **без нито един `for`/`while` цикъл** извън comprehensions – само с comprehensions, `map`, `filter`, `functools.reduce`, `sorted`, `zip`, `any`, `all`, `sum`, `max` и т.н.

* `revenue_by_customer(orders)` – речник `клиент → общ приход` (`quantity * price`)
* `top_products(orders, n)` – списък от `n` наредени двойки `(продукт, общо количество)` с най-голямо количество; при равенство – по име възходящо
* `loyal_customers(orders)` – сортиран списък от клиентите, които имат поръчка във **всеки** от месеците, които се срещат в данните
* `monthly_growth(orders)` – списък от тройки `(месец, приход, промяна)` по месеци възходящо, където `промяна` е процентната промяна спрямо предходния месец, закръглена до 1 знак (`None` за първия месец)
* `best_month_per_product(orders)` – речник `продукт → месецът с най-голямо количество продадени` (при равенство – по-ранният месец)

In [ ]:
from collections import namedtuple
from functools import reduce

Order = namedtuple("Order", ["customer", "product", "quantity", "price", "month"])


def revenue_by_customer(orders):
    pass  # your code here


def top_products(orders, n):
    pass  # your code here


def loyal_customers(orders):
    pass  # your code here


def monthly_growth(orders):
    pass  # your code here


def best_month_per_product(orders):
    pass  # your code here


# Tests:

orders = [
    Order("ana", "coffee", 2, 5.0, 1),
    Order("bob", "tea", 3, 3.0, 1),
    Order("ana", "cake", 1, 8.0, 2),
    Order("cvetan", "coffee", 4, 5.0, 2),
    Order("bob", "coffee", 1, 5.0, 2),
    Order("ana", "tea", 5, 3.0, 3),
    Order("bob", "cake", 2, 8.0, 3),
    Order("cvetan", "tea", 1, 3.0, 3),
]

assert revenue_by_customer(orders) == {"ana": 33.0, "bob": 30.0, "cvetan": 23.0}
assert revenue_by_customer([]) == {}

assert top_products(orders, 2) == [("tea", 9), ("coffee", 7)]
assert top_products(orders, 10) == [("tea", 9), ("coffee", 7), ("cake", 3)]
assert top_products(orders + [Order("ana", "beer", 7, 2.0, 3)], 3) == [("tea", 9), ("beer", 7), ("coffee", 7)]

assert loyal_customers(orders) == ["ana", "bob"]
assert loyal_customers(orders[:2]) == ["ana", "bob"]
assert loyal_customers([]) == []

assert monthly_growth(orders) == [(1, 19.0, None), (2, 33.0, 73.7), (3, 34.0, 3.0)]
assert monthly_growth(orders[:1]) == [(1, 10.0, None)]

assert best_month_per_product(orders) == {"coffee": 2, "tea": 3, "cake": 3}
assert best_month_per_product(orders + [Order("bob", "cake", 1, 8.0, 2)]) == {"coffee": 2, "tea": 3, "cake": 2}, "earlier month wins ties"

"✅ All OK!"